In [9]:
import pandas as pd

df = pd.read_csv("../assets/synthetic-data-prices.csv")

df.head(10)

,ticker,date,open,high,low,close,volume
0,AAPL,2025-01-02,175.00,176.70,173.05,176.46,29979419
1,AMZN,2025-01-02,185.00,187.37,183.49,186.49,75756103
2,MSFT,2025-01-02,415.00,424.21,412.82,422.10,64023381
3,NVDA,2025-01-02,480.00,483.70,478.66,481.43,43396306
4,TSLA,2025-01-02,240.00,243.41,235.20,235.84,36117675
5,AAPL,2025-01-03,176.46,176.89,171.84,172.64,51598928
6,AMZN,2025-01-03,186.49,191.39,185.35,191.06,70656608
7,MSFT,2025-01-03,422.10,438.47,420.01,434.65,25388239
8,NVDA,2025-01-03,481.43,501.77,479.30,494.84,61296233
9,TSLA,2025-01-03,235.84,237.72,231.16,231.77,58141058


In [10]:
# For OpenAI tool use, we need to define a function schema for serialization (not the function itself).
# Provide an OpenAI-compatible JSON schema for the function.

# The original local helper function (still needed for tool execution):
def lookup_price(ticker: str, date: str, fields=None, dataframe=None):
    """
    Simple function to look up price data for a given ticker and date.

    Args:
        ticker (str): Stock ticker (case-insensitive).
        date (str or datetime): Date as 'YYYY-MM-DD' or datetime.
        fields (list or None): Specific columns to return; if None, returns all.
        dataframe (pd.DataFrame or None): DataFrame to use; if None, uses 'df' in global scope.

    Returns:
        dict or None: Result dict or None if not found.
    """
    df_use = dataframe if dataframe is not None else df
    date_pd = pd.to_datetime(date)
    row = df_use[
        (df_use['ticker'].str.upper() == ticker.upper()) &
        (pd.to_datetime(df_use['date']) == date_pd)
    ]
    if row.empty:
        return None
    record = row.iloc[0]
    if fields is None:
        return record.to_dict()
    else:
        return {field: record[field] for field in fields if field in record}


# Example usage:
lookup_price("AAPL", "2025-01-03", fields=["open", "close"])

{'open': np.float64(176.46), 'close': np.float64(172.64)}

In [14]:
import json

# Responses API uses a FLAT tool schema (no nested "function" wrapper).
# Compare with Chat Completions API which nests everything under "function": { ... }
lookup_price_openai_tool = {
    "type": "function",
    "name": "lookup_price",
    "description": "Look up price data for a given ticker and date.",
    "parameters": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker (case-insensitive)"
            },
            "date": {
                "type": "string",
                "description": "Date as 'YYYY-MM-DD'"
            },
            "fields": {
                "type": "array",
                "items": {
                    "type": "string"
                },
                "description": "Specific columns to return (e.g. ['open', 'close'])"
            }
        },
        "required": ["ticker", "date"],
    },
}

In [15]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5",
    instructions="""
    You are a helpful assistant that can answer questions and help with tasks.
    You use the tool: 'lookup_price' to get price data for a given ticker and date 
    when the user requests.
    """,
    input="What was the price of AAPL on 2025-01-03?",
    tools=[lookup_price_openai_tool],
)

response

Response(id='resp_0ed2046908516fb30069aaf70b66dc81948559c40d09c016c3', created_at=1772812043.0, error=None, incomplete_details=None, instructions="\n    You are a helpful assistant that can answer questions and help with tasks.\n    You use the tool: 'lookup_price' to get price data for a given ticker and date \n    when the user requests.\n    ", metadata={}, model='gpt-5-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0ed2046908516fb30069aaf70bf2708194bbb563adfa1b0a5e', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"ticker":"AAPL","date":"2025-01-03","fields":["open","high","low","close","volume"]}', call_id='call_w5O0l3iXwtaaMKs3D6xbXZfJ', name='lookup_price', type='function_call', id='fc_0ed2046908516fb30069aaf71014c08194b64e6ed7110bb4e4', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='lookup_price', parameters={'type': 'objec

In [16]:
# Map of available tool functions
tool_functions = {
    "lookup_price": lookup_price
}

# Build the conversation from the first response's output
input_messages = list(response.output)

# Execute any function_call items and append results
for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = tool_functions[item.name](**args)
        input_messages.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result, default=str)  # default=str handles numpy types
        })

# Send the tool results back to get a natural language answer
final_response = client.responses.create(
    model="gpt-4o-mini",
    instructions="""
    You are a helpful assistant that can answer questions and help with tasks.
    You use the tool: 'lookup_price' to get price data for a given ticker and date 
    when the user requests.
    """,
    input=input_messages,
    tools=[lookup_price_openai_tool],
)

print(final_response.output_text)

On January 3, 2025, Apple Inc. (AAPL) had the following price data:

- **Open:** $176.46
- **High:** $176.89
- **Low:** $171.84
- **Close:** $172.64
- **Volume:** 51,598,928 shares

If you need more information or additional data, feel free to ask!
